# Robot Synthetic Data Generation Workshop (RDNA4/3.5)

## Background & Motivation

**The data scarcity problem** is the fundamental bottleneck in robot learning. Unlike language models trained on internet-scale text, or vision models trained on billions of images, robot learning requires physical interaction with the real world — expensive, slow, and hard to scale. A single demonstration episode might take 30 seconds, but robust policy training demands thousands of such episodes.

**Synthetic Data Generation (SDG)** addresses this by using physics simulation to generate unlimited expert demonstrations automatically. Among the four main SDG pathways — (A) Physics Simulation, (B) Video Reconstruction, (C) World Models, (D) Data Augmentation — **physics simulation** offers ground-truth state/action data, unlimited scale, domain randomization, and safe exploration.

This workshop implements a complete **end-to-end Sim → Train → Eval** pipeline on AMD RDNA GPUs (ROCm) using the [Genesis](https://genesis-embodied-ai.github.io/) simulation engine and the [LeRobot](https://github.com/huggingface/lerobot) framework:

```
┌──────────────────────────────┐  ┌─────────────────────┐  ┌──────────────────────────┐
│ 02_gen_data_custom_scene.py   │  │  02_train_vla.py     │  │  04_eval_custom_scene.py  │
│ kitchen GLB + floor_origin    │─▶│  SmolVLA fine-tune   │─▶│  Closed-loop eval         │
│ 100 episodes, GPU render      │  │  freeze vision enc,  │  │  GPU render (radeonsi)    │
│ 2 cams: up + wrist            │  │  train expert + proj │  │  success rate + video     │
└──────────────────────────────┘  └─────────────────────┘  └──────────────────────────┘
     ~15 min                          ~7-11 min                    ~4 min
```

**Workshop routing**: this notebook runs on **RDNA4 (R9700)** or **RDNA3.5 (W7900)** nodes. Unlike the CDNA3 path, **all three steps execute end-to-end during the workshop** — data is generated locally using GPU hardware rasterization (radeonsi), no HuggingFace download needed. Total wall time: **~30 min**.

## Data Flow

```
Genesis Scene (Kitchen)          LeRobot Dataset                SmolVLA
┌──────────────┐                ┌──────────────┐              ┌──────────────┐
│ Franka Panda │                │ observation   │              │ Vision       │
│ Red Cube     │──IK plan──────▶│  .state [9D]  │──train──────▶│ Encoder      │
│ Kitchen GLB  │   joint lerp   │  .images.up   │              │ (frozen)     │
│ 2 Cameras:   │   render(GPU)  │  .images.side │              │              │
│   up + wrist │                │ action [9D]   │              │ Expert       │
│ Physics sim  │                │ task (text)   │              │ Layers       │
│ (Genesis)    │                │              │              │ (trainable)  │
└──────────────┘                └──────────────┘              └──────────────┘
```

## Workshop Steps

| Step | Script | Time | Description |
|------|--------|------|-------------|
| **0** | Setup | — | Check GPU, verify deps, download kitchen assets |
| **1** | `02_gen_data_custom_scene.py` | ~15 min | Generate 100 episodes in kitchen scene (GPU render) |
| **2** | `02_train_vla.py` | ~7-11 min | Fine-tune SmolVLA (~450M params) on local dataset |
| **3** | `04_eval_custom_scene.py` | ~4 min | Closed-loop evaluation in kitchen scene (GPU render) |

---
## 0. Environment Setup

This section verifies the runtime environment:

- **GPU detection** — verifies CUDA/ROCm availability and VRAM capacity
- **Dependency check** — `genesis-world`, `lerobot`, `transformers`

> **Hardware**: verified on **RDNA4 (Radeon AI PRO R9700, ROCm 7.2)** and **RDNA3.5 (Radeon PRO W7900D, ROCm 7.0/7.2)**. Genesis uses GPU hardware rasterization (radeonsi) for both data generation and evaluation — no CPU render fallback needed. Kitchen GLB assets are downloaded automatically in Section 1.

In [ ]:
import os, sys, json, subprocess, time
from pathlib import Path
from IPython.display import display, Image, HTML, Video
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
matplotlib.rcParams['figure.dpi'] = 120

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

WORKSHOP_DIR = Path(os.getcwd())
SCRIPTS_DIR = WORKSHOP_DIR / 'scripts'
OUTPUT_DIR = WORKSHOP_DIR / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Workshop dir: {WORKSHOP_DIR}')
print(f'Output dir:   {OUTPUT_DIR}')
print(f'Python:       {sys.executable}')

### 0.1 GPU & ROCm Check

In [ ]:
import torch

print(f'PyTorch:  {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / 1024**3
        print(f'  GPU[{i}]: {props.name}  VRAM: {vram_gb:.1f} GB')
else:
    print('WARNING: No GPU detected.')

!rocm-smi --showuse 2>/dev/null | head -20 || echo 'rocm-smi not available'

In [ ]:
import lerobot
print(f'lerobot: {lerobot.__version__}')

try:
    import genesis as gs
    print(f'genesis: {gs.__version__}')
except Exception as e:
    print(f'genesis import: {e}')

### 0.2 Verify Genesis

In [ ]:
print(f'Genesis version: {gs.__version__}')
print(f'All core packages loaded successfully.')
print('Environment ready for end-to-end pipeline.')

In [ ]:
import torchcodec
print(f'torchcodec: {torchcodec.__version__}')
print(f'torch: {torch.__version__}')
print(f'GPU render backend: radeonsi (hardware rasterization)')
print('\nAll dependencies verified. Ready to generate data.')

---
## 1. Synthetic Data Generation — Kitchen Scene (100 episodes, ~15 min)

On RDNA4/3.5 nodes, we generate the full training dataset locally using GPU hardware rasterization. This is the key advantage of RDNA over CDNA3 — **3-4× faster rendering** with no CPU fallback.

This step uses **inverse kinematics (IK) planning** inside the Genesis physics simulator to generate expert-quality pick-cube trajectories. The IK planner produces task-space waypoints (hover → descend → grasp → lift), solves for joint angles, and interpolates between them.

We use the **Rustic Kitchen** 3D scene (GLB mesh from [World Labs Marble](https://marble.worldlabs.ai/)) with the `floor_origin` anchor point, and an **up + wrist** camera layout that includes an eye-in-hand camera mounted on the robot's end-effector. This setup produces richer visual features than a flat plane and matches the CDNA3 pre-built dataset on HuggingFace.

### Why IK-based SDG?

| Paradigm | Logic | Smoothness | Recovery |
|----------|-------|-----------|----------|
| **IK Open-loop** ← (this workshop) | One-shot IK solve → joint-space interpolation | Excellent | None |
| **IK Closed-loop** | Re-solve IK from current config each step | Good | Implicit |
| **RL Rollout** | Policy infers action from observation | Variable | Explicit |

### Configuration

- **Robot**: Franka Panda (9 DOF: 7 arm joints + 2 finger joints)
- **Task**: Pick up a red cube from randomized XY positions
- **Scene**: Rustic Kitchen (GLB mesh), `floor_origin` anchor
- **Trajectory**: `HOME → hover → descend → grasp → lift`
- **Cameras**: `up` (overhead) + `wrist` (eye-in-hand), 640×480 RGB, GPU rendered (radeonsi)
- **Output**: LeRobot-format dataset (100 ep, ~13,500 frames, AV1 video)
- **Expected time**: ~15 min (W7900D) / ~23 min (R9700)

### 1.0 Download Kitchen Assets

The kitchen scene requires GLB mesh files (~130 MB). The pre-built Docker image may already have them cached; this cell downloads them if missing.

In [ ]:
!python {SCRIPTS_DIR}/00_download_kitchen.py --mesh-only

### 1.1 Generate 100 Episodes

Running full data generation in the kitchen scene with up+wrist cameras. The first episode includes Taichi kernel compilation (~30s overhead), subsequent episodes run at steady-state speed (~9-12 s/ep).

In [ ]:
N_EPISODES = 100
DATASET_ID = 'local/franka-kitchen-wrist-100ep'

t0 = time.time()
!python {SCRIPTS_DIR}/02_gen_data_custom_scene.py \
    --scene rustic_kitchen --anchor floor_origin \
    --camera-layout up_wrist \
    --n-episodes {N_EPISODES} \
    --repo-id {DATASET_ID} \
    --seed 42

elapsed = time.time() - t0
print(f'\nData generation complete: {N_EPISODES} episodes in {elapsed:.0f}s ({elapsed/60:.1f} min)')
print(f'Per-episode: {elapsed/N_EPISODES:.1f} s/ep')

### 1.2 Load and Visualize the Generated Dataset

The dataset is now stored locally. Let's load it and visualize camera views + joint trajectories.

In [ ]:
try:
    from lerobot.common.datasets.lerobot_dataset import LeRobotDataset
except ImportError:
    from lerobot.datasets.lerobot_dataset import LeRobotDataset

ds = LeRobotDataset(DATASET_ID)
print(f'Dataset: {DATASET_ID}')
print(f'  Episodes:     {ds.num_episodes}')
print(f'  Total frames: {len(ds)}')
print(f'  FPS:          {ds.fps}')
print(f'  Features:     {list(ds.features.keys())}')

sample = ds[0]
print(f'\nSample keys: {list(sample.keys())}')
for k, v in sample.items():
    if hasattr(v, 'shape'):
        print(f'  {k}: shape={v.shape} dtype={v.dtype}')
    elif isinstance(v, str):
        print(f'  {k}: "{v}"')
    else:
        print(f'  {k}: {type(v).__name__} = {v}')

#### Camera Views

The plot below shows 6 evenly-spaced frames from episode 0 for each camera:

- **`observation.images.up`** — overhead world-fixed view (top row)
- **`observation.images.side`** — wrist-mounted eye-in-hand camera (bottom row)

> Note: The tensor key is named `side` for backward compatibility, but in the `up_wrist` camera layout it is a wrist camera attached to the robot's end-effector.

In [ ]:
# Visualize camera images from a sample episode
from PIL import Image as PILImage

def get_episode_range(dataset, ep_idx):
    if hasattr(dataset, 'episode_data_index'):
        start = dataset.episode_data_index['from'][ep_idx].item()
        end = dataset.episode_data_index['to'][ep_idx].item()
    else:
        fpe = len(dataset) // dataset.num_episodes
        start, end = ep_idx * fpe, (ep_idx + 1) * fpe
    return start, end

def show_episode_frames(dataset, ep_idx=0, n_frames=6):
    start, end = get_episode_range(dataset, ep_idx)
    indices = np.linspace(start, end - 1, n_frames, dtype=int)
    img_keys = [k for k in dataset.features if 'images' in k]
    if not img_keys:
        print('No image features found')
        return

    fig, axes = plt.subplots(len(img_keys), n_frames,
                             figsize=(3 * n_frames, 3 * len(img_keys)))
    if len(img_keys) == 1:
        axes = axes[np.newaxis, :]

    for row, key in enumerate(img_keys):
        for col, idx in enumerate(indices):
            sample = dataset[int(idx)]
            img = sample[key]
            if hasattr(img, 'numpy'):
                img = img.numpy()
            if img.ndim == 3 and img.shape[0] in (1, 3):
                img = np.transpose(img, (1, 2, 0))
            if img.dtype != np.uint8:
                if img.max() <= 1.0:
                    img = (img * 255).astype(np.uint8)
                else:
                    img = img.astype(np.uint8)
            axes[row, col].imshow(img)
            axes[row, col].set_title(f'frame {idx - start}', fontsize=8)
            axes[row, col].axis('off')
        axes[row, 0].set_ylabel(key.split('.')[-1], fontsize=10)

    fig.suptitle(f'Episode {ep_idx} Camera Views', fontsize=12)
    plt.tight_layout()
    fig.savefig(str(OUTPUT_DIR / f'ep{ep_idx}_camera_views.png'), bbox_inches='tight')
    plt.show()
    print(f'Saved: {OUTPUT_DIR / f"ep{ep_idx}_camera_views.png"}')

show_episode_frames(ds, ep_idx=0)

#### Joint Trajectories

The 3x3 grid below plots `observation.state` (blue) vs `action` (orange) for each of the 9 DOFs across episode 0. Joints `j1`–`j7` are the arm joints; `f1`–`f2` are the finger (gripper) joints.

- **state** = actual joint position read from the simulator (where the robot *is*)
- **action** = commanded target joint position (where we want the robot to *go*)

In IK-planned open-loop trajectories, the action–state gap is typically very small because the PD controller closely tracks the commanded positions.

In [ ]:
# Plot joint trajectories for episode 0
def plot_joint_trajectory(dataset, ep_idx=0):
    start, end = get_episode_range(dataset, ep_idx)
    states, actions = [], []
    for i in range(start, end):
        s = dataset[i]
        states.append(s['observation.state'].numpy())
        actions.append(s['action'].numpy())

    states = np.array(states)
    actions = np.array(actions)
    joint_names = ['j1','j2','j3','j4','j5','j6','j7','f1','f2']

    fig, axes = plt.subplots(3, 3, figsize=(14, 8), sharex=True)
    for i, ax in enumerate(axes.flat):
        if i >= states.shape[1]:
            ax.set_visible(False)
            continue
        ax.plot(states[:, i], label='state', linewidth=1)
        ax.plot(actions[:, i], label='action', linewidth=1, alpha=0.7)
        name = joint_names[i] if i < len(joint_names) else f'dim{i}'
        ax.set_title(name, fontsize=9)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

    fig.suptitle(f'Episode {ep_idx} Joint Trajectories (state vs action)', fontsize=12)
    plt.tight_layout()
    fig.savefig(str(OUTPUT_DIR / f'ep{ep_idx}_joint_trajectory.png'), bbox_inches='tight')
    plt.show()

plot_joint_trajectory(ds, ep_idx=0)

---
## 2. SmolVLA Post-Training

### What is SmolVLA?

[SmolVLA](https://huggingface.co/blog/smolvla) is a **Vision-Language-Action** model that combines a visual encoder (SigLIP), a language model (SmolLM2), and an action expert to predict robot actions from images, proprioceptive state, and a natural-language task description.

| Component | Details |
|-----------|---------|
| **Architecture** | VLM (SigLIP + SmolLM2) + Action Expert |
| **Inputs** | images (up + side) + state (9D) + language task |
| **Output** | action chunk (50 × 9D joint targets) |
| **Total params** | ~450M |
| **Trainable params** | ~5M (expert only, vision encoder frozen) |

### Why SmolVLA for SDG?

Compared to simpler policies (MLP BC, ACT), SmolVLA brings:
- **Language conditioning** — the task instruction "Pick up the red cube" guides behavior
- **Pretrained vision** — SigLIP encoder provides strong visual features without training from scratch
- **Efficient fine-tuning** — only the action expert layers are trained, keeping VRAM under 4 GB

### Training Setup

We fine-tune `lerobot/smolvla_base` on the **locally generated** dataset from Section 1:
- `chunk_size=50`, `n_action_steps=50`
- AdamW optimizer, `freeze_vision_encoder=True`
- Training only the expert layers + state projection head (~99.9M trainable / 450M total)

**Default recipe**: `--batch-size 4 --num-workers 4`. AMP BF16 + PyTorch SDPA auto-dispatch (AOTriton flash on AMD) are auto-enabled.

**Expected time**: ~7.4 min (R9700) / ~10.6 min (W7900D).

In [ ]:
N_TRAIN_STEPS = 4000
BATCH_SIZE = 4
RUN_NAME = 'smolvla_kitchen_wrist_rdna'
SAVE_DIR = OUTPUT_DIR / 'train' / RUN_NAME

SMOLVLA_LOCAL = '/opt/workshop/models/smolvla_base'
VLM_LOCAL = '/opt/workshop/models/SmolVLM2-500M-Video-Instruct'

# Patch lerobot VLM 路径指向本地 (regex 替换完整路径，避免子串拼接)
import site, re
site_pkg = site.getsitepackages()[0]
for pyfile in ['policies/smolvla/configuration_smolvla.py',
               'policies/smolvla/smolvlm_with_expert.py']:
    fpath = Path(site_pkg) / 'lerobot' / pyfile
    if not fpath.exists():
        continue
    src = fpath.read_text()
    if VLM_LOCAL in src and src.count(VLM_LOCAL) == src.count('SmolVLM2-500M-Video-Instruct'):
        print(f'[ok] {pyfile}')
    else:
        fixed = re.sub(r'["\'][^"\']*SmolVLM2-500M-Video-Instruct["\']',
                        f'"{VLM_LOCAL}"', src)
        if fixed != src:
            fpath.write_text(fixed)
            print(f'[patch] {pyfile} → local')
        else:
            print(f'[skip] {pyfile}')

t0 = time.time()
!python {SCRIPTS_DIR}/02_train_vla.py \
    --dataset-id {DATASET_ID} \
    --pretrained {SMOLVLA_LOCAL} \
    --n-steps {N_TRAIN_STEPS} \
    --batch-size {BATCH_SIZE} \
    --num-workers 4 \
    --output-dir {OUTPUT_DIR} --run-name {RUN_NAME}

elapsed = time.time() - t0
print(f'\nTraining complete in {elapsed:.0f}s ({elapsed/60:.1f} min)')

### 2.1 Training Loss Curve

The loss curve shows the MSE loss between predicted and ground-truth action chunks over training steps. The gradient norm plot (right) monitors training stability — spikes may indicate learning rate issues or data batch outliers.

**Important caveat**: A decreasing loss does *not* guarantee a working policy. As observed in our experiments with SmolVLA, the model can achieve low training loss by overfitting to specific training images while failing to generalize in closed-loop evaluation, where rendered images differ from the training distribution (especially with `freeze_vision_encoder=True`).

In [ ]:
def plot_loss_curve(save_dir):
    metrics_path = save_dir / 'train_metrics.json'
    summary_path = save_dir / 'train_summary.json'
    if not metrics_path.exists():
        print(f'Metrics not found: {metrics_path} (training may not have completed)')
        return

    metrics = json.loads(metrics_path.read_text())
    summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}

    steps = [m['step'] for m in metrics]
    losses = [m['loss'] for m in metrics]
    grad_norms = [m['grad_norm'] for m in metrics]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(steps, losses, linewidth=0.8, alpha=0.5, label='per-step')
    window = max(1, len(losses) // 20)
    if len(losses) > window:
        smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
        ax1.plot(steps[window-1:], smoothed, linewidth=2, color='#e74c3c', label=f'smooth(w={window})')
    ax1.set_xlabel('Step'); ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.plot(steps, grad_norms, linewidth=0.8, alpha=0.6, color='#3498db')
    ax2.set_xlabel('Step'); ax2.set_ylabel('Grad Norm')
    ax2.set_title('Gradient Norm'); ax2.grid(True, alpha=0.3)

    info = ''
    if summary:
        info = (f"loss: {summary.get('loss_start','?'):.4f} -> {summary.get('loss_end','?'):.4f}  "
                f"time: {summary.get('total_time_s',0):.0f}s  VRAM: {summary.get('peak_vram_mb',0):.0f}MB")
    fig.suptitle(f'SmolVLA Training  |  {info}', fontsize=10)
    plt.tight_layout()
    fig.savefig(str(save_dir / 'loss_curve.png'), bbox_inches='tight')
    plt.show()

plot_loss_curve(SAVE_DIR)

---
## 3. Simulation Evaluation — Kitchen Scene (~4 min)

### Closed-loop Evaluation Loop

At each simulation step, the evaluation script runs:

1. **Render** `up` + `wrist` camera images via GPU (radeonsi) in the kitchen scene — same visual domain as training data
2. **Read** current joint angles as `observation.state`
3. **Infer** action chunk via SmolVLA (images + state + task text)
4. **Execute** first action via PD position control
5. **Step** Genesis physics
6. **Check** success: cube lifted ≥2cm and sustained ≥8 frames

### Key Advantage of RDNA (GPU render)

Since both data generation and evaluation use the same GPU rasterizer (radeonsi) and the same kitchen scene, there is **no render-gap bias**. This is why RDNA nodes produce benchmark-quality eval numbers, unlike CDNA3 where CPU rendering introduces a ~20 pt systematic bias.

> **Note**: Eval requires a trained checkpoint from Step 2. The eval script (`04_eval_custom_scene.py`) must use the same scene and camera layout as data generation.

In [ ]:
CHECKPOINT = SAVE_DIR / 'final'
EVAL_DIR = OUTPUT_DIR / 'eval' / 'kitchen_eval'

if CHECKPOINT.exists():
    t0 = time.time()
    !python {SCRIPTS_DIR}/04_eval_custom_scene.py \
        --checkpoint {CHECKPOINT} \
        --dataset-id {DATASET_ID} \
        --scene rustic_kitchen --anchor floor_origin \
        --camera-layout up_wrist \
        --n-episodes 20 --max-steps 150 --seed 99 \
        --record-video \
        --output-dir {OUTPUT_DIR}
    elapsed = time.time() - t0
    print(f'\nEvaluation complete in {elapsed:.0f}s ({elapsed/60:.1f} min)')
else:
    print(f'No checkpoint at {CHECKPOINT} - skipping eval (training may not have completed)')

In [ ]:
eval_summary_path = EVAL_DIR / 'eval_summary.json'
if eval_summary_path.exists():
    summary = json.loads(eval_summary_path.read_text())
    print('=== Evaluation Results ===')
    for k, v in summary.items():
        print(f'  {k}: {v}')

    eval_videos = sorted(EVAL_DIR.rglob('*.mp4'))
    if eval_videos:
        print(f'\nEval videos ({len(eval_videos)}):')
        for v in eval_videos[:3]:
            print(f'  {v.name}')
            try:
                display(Video(str(v), embed=True, width=640))
            except Exception as e:
                print(f'  (cannot embed video: {e})')
else:
    print('No eval summary found — training/eval may not have completed.')

---
## 4. Summary & Artifacts

### What We Accomplished (End-to-End on RDNA)

| Step | What | Time | Key Output |
|------|------|------|-----------|
| **Data Generation** | 100 episodes in kitchen scene, up+wrist cameras, GPU-rendered | ~15 min | LeRobot dataset (local) |
| **VLA Training** | SmolVLA fine-tune, 4000 steps | ~7-11 min | Checkpoint + loss curve |
| **Evaluation** | 20 ep closed-loop in kitchen scene, GPU render | ~4 min | Success rate + videos |
| **Total** | Full pipeline | **~30 min** | |

In [ ]:
print('\nWorkshop pipeline complete!')
print(f'All outputs saved to: {OUTPUT_DIR}')

---
## References & Further Reading

### Frameworks
- [LeRobot](https://github.com/huggingface/lerobot) — Robot learning framework by HuggingFace (dataset format, policies, training)
- [Genesis](https://genesis-embodied-ai.github.io/) — GPU-accelerated physics simulation engine (ROCm compatible via Taichi)
- [SmolVLA](https://huggingface.co/blog/smolvla) — Vision-Language-Action model for robot control

### SDG & Imitation Learning
- [DART](https://proceedings.mlr.press/v78/laskey17a.html) (CoRL 2017) — Noise injection with closed-loop supervisor for reducing covariate shift
- [MimicGen](https://mimicgen.github.io/) (CoRL 2023) — Few demonstrations → large-scale data via motion replanning
- [Domain Randomization for Sim-to-Real Transfer](https://arxiv.org/abs/1703.06907) — Visual/physical randomization for transfer

### VLA Research (2025–2026)
- [DiffRL→VLA](https://arxiv.org/abs/2509.19752) — Diffusion RL for smoother rollout trajectories
- [SimpleVLA-RL](https://arxiv.org/abs/2509.09674) (ICLR 2026) — Direct RL fine-tuning of VLA models
- [RL-Co](https://arxiv.org/abs/2602.12628) — SFT warm-start + sim RL fine-tune + real data anti-forgetting
- [DLR](https://arxiv.org/abs/2511.19528) — Diverse RL trajectories for VLA pretraining

### AMD ROCm
- [AMD ROCm Documentation](https://rocm.docs.amd.com/)
- [PyTorch ROCm Installation](https://pytorch.org/get-started/locally/)
- [World Labs Marble](https://marble.worldlabs.ai/) — 3D scene generation for custom simulation environments